# Dynamic Resource Allocation (DRA)

A practical reference for **Dynamic Resource Allocation (DRA)** — the Kubernetes
API for requesting, configuring, and sharing specialized hardware such as GPUs.
DRA replaces the rigid, count-only *device plugin* model with first-class
Kubernetes objects (`DeviceClass`, `ResourceClaim`, `ResourceSlice`) that let
workloads ask for devices by *attributes* — "a GPU with ≥ 40 GB of memory", "a
MIG slice", "two GPUs on the same NVLink domain" — and let drivers like the
**NVIDIA DRA Driver for GPUs** satisfy those requests with fine-grained sharing.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

### What is it?

**Dynamic Resource Allocation (DRA)** is a Kubernetes framework (API group
`resource.k8s.io`) for requesting and sharing devices. It models hardware as
structured, schedulable objects instead of opaque "extended resources" counted
as integers. The core DRA feature graduated to **GA in Kubernetes 1.34**, after
a long beta as the *structured-parameters* design (beta in **1.32**); several
companion features (admin access, prioritized lists, partitionable devices,
consumable capacity) are still alpha/beta and gated behind their own feature
flags.

A device driver runs on each node and publishes **`ResourceSlice`** objects that
advertise the devices it manages and their **attributes** (model, memory,
compute capability, MIG profile, NVLink domain, …). A workload author writes a
**`ResourceClaim`** (or a **`ResourceClaimTemplate`** that mints one claim per
pod) whose *device requests* select devices with **CEL expressions** over those
attributes. The **kube-scheduler** evaluates the claim against available
ResourceSlices, picks devices that satisfy it *and* fit the pod's other
constraints, and records the allocation back on the claim. The driver then
**prepares** the chosen devices on the node (e.g. via CDI) so the container sees
them.

### Why use it?

Key benefits of using DRA:

- **Request by attribute, not by count.** Ask for "a GPU with ≥ 80 GB" or "an
  H100", and the scheduler matches it — instead of hard-coding
  `nvidia.com/gpu: 1` and hoping every node is identical.
- **Real, declarative GPU sharing.** A single claim can be shared by multiple
  pods/containers; the NVIDIA driver implements time-slicing, **MPS**, and
  **dynamic MIG** through claim configuration rather than node-level static
  config.
- **Topology- and capacity-aware allocation.** Match constraints ("all GPUs in
  one NVLink/IMEX domain") and (alpha) consumable capacity let you co-schedule
  tightly-coupled work and pack fractional devices.
- **Driver-owned configuration.** Vendor-specific knobs travel *with the claim*
  (opaque `config`), so cluster admins don't pre-bake every GPU mode into the
  node.

### When to use it?

DRA is particularly useful when:

- Your cluster has **heterogeneous accelerators** and workloads need to pick the
  right one by capability rather than by node label.
- You want **fractional / shared GPU** access (time-slicing, MPS, dynamic MIG)
  driven per-workload instead of per-node.
- You run **multi-GPU / multi-node** jobs that need topology guarantees such as
  a shared NVLink domain (e.g. GB200 NVL72 via IMEX).

## Key Features

### Core Capabilities of DRA

| Capability | Description | Why it matters |
|---|---|---|
| Attribute-based selection | Requests use CEL over device `attributes`/`capacity` published in `ResourceSlice`s | Ask for what a device *is*, not just how many you want |
| `DeviceClass` | Cluster-scoped category + base selectors/config for a kind of device | Admins curate the menu; users reference a class by name |
| `ResourceClaim` / `…Template` | Namespaced request for one or more devices; template mints one claim per pod | Per-pod isolation or explicit cross-pod sharing |
| `ResourceSlice` | Driver-published inventory of devices and their attributes per node/pool | The scheduler's source of truth; no out-of-band device plugin counts |
| Structured parameters | Scheduler understands claims natively (no vendor scheduler webhook) | Reliable, racefree scheduling and `nodeSelector` integration |
| Device sharing | A claim can be consumed by multiple pods/containers | Enables time-slicing / MPS / fractional GPUs |
| Match attributes & constraints | Require selected devices to agree on an attribute (e.g. same NVLink domain) | Topology-aware multi-GPU placement |
| Opaque driver config | Vendor config (`sharing`, MIG profile, MPS) carried on the claim/class | Per-workload device mode without node reconfiguration |

## Architecture Overview

```text
  Cluster (control plane)                       GPU node
  ─────────────────────────                     ─────────────────────────────
   DeviceClass        (admin-curated)            NVIDIA driver + NVML + (MIG)
        ▲                                                 ▲
        │ references                                      │ enumerates devices
   ResourceClaim / ResourceClaimTemplate                  │
        │  device requests (CEL selectors)         ┌───────┴────────────────┐
        ▼                                          │  DRA driver (DaemonSet) │
   kube-scheduler ──── reads ───► ResourceSlice ◄──┤  • kubelet plugin       │
        │  picks devices, writes allocation        │  • publishes ResourceSlice
        │  result back onto the claim              │  • NodePrepareResources │
        ▼                                          └───────┬────────────────┘
   Pod (spec.resourceClaims +                              │ CDI injects devices
        container resources.claims) ─── kubelet ──────────►│ container sees GPU(s)
```

### Components

1. **`DeviceClass`** — cluster-scoped object defining a category of devices
   (e.g. `gpu.nvidia.com`) plus default selectors and config. Claims reference a
   class by name.
2. **`ResourceClaim`** — a namespaced request for one or more devices. Each
   *request* names a `DeviceClass` and adds CEL `selectors`; an `allocationMode`
   of `ExactCount` or `All` controls how many devices are claimed.
3. **`ResourceClaimTemplate`** — embedded in a pod/Deployment so the controller
   stamps out a **fresh claim per pod**; the usual choice for workloads.
4. **`ResourceSlice`** — published by the driver, advertising the devices in a
   *pool* on a node and their attributes/capacity. The scheduler reads these.
5. **DRA driver (DaemonSet)** — the vendor component. It runs a **kubelet
   plugin** that publishes ResourceSlices and implements
   `NodePrepareResources` / `NodeUnprepareResources`, wiring chosen devices into
   containers (NVIDIA uses **CDI**).
6. **kube-scheduler** — with structured parameters it evaluates claims against
   ResourceSlices, allocates devices, and records the result on the claim's
   `status.allocation`.

## Installation

### Prerequisites

- **Kubernetes ≥ 1.34** for GA core DRA (or ≥ 1.32 with the `DynamicResourceAllocation`
  feature gate enabled on the API server, scheduler, controller-manager, and
  kubelet). Alpha sub-features need their own gates (e.g.
  `DRAAdminAccess`, `DRAPartitionableDevices`, `DRAConsumableCapacity`).
- The `resource.k8s.io/v1` API group enabled (it is, by default, on GA clusters).
- **GPU nodes** with the NVIDIA datacenter driver and the **NVIDIA Container
  Toolkit** configured with **CDI** enabled.
- For NVIDIA GPU DRA: the **NVIDIA DRA Driver for GPUs**
  (`nvidia-dra-driver-gpu`), installed via Helm. The **GPU Operator** can manage
  the prerequisite driver/toolkit stack.

### Installation steps

> DRA is a **Kubernetes feature plus a vendor driver**, not a Python/pip
> package. The cell below shows the real commands; there is nothing to
> `pip install`.

In [ ]:
%%bash
# DRA core is a Kubernetes API; you install a *vendor driver* for your hardware.
# Below: enable the gate (pre-1.34) and install the NVIDIA DRA driver via Helm.

# 1) (Only needed before K8s 1.34) enable the feature gate on every component.
#    On a kubeadm cluster, add to the API server / scheduler / controller-manager
#    manifests and the kubelet config:
#      --feature-gates=DynamicResourceAllocation=true
#      --runtime-config=resource.k8s.io/v1=true   # API server only
#
#    Confirm the API group is served:
kubectl api-resources --api-group=resource.k8s.io

# 2) Verify CDI is on for the NVIDIA Container Toolkit on each GPU node:
#    /etc/nvidia-container-runtime/config.toml  ->  [nvidia-container-runtime] mode = "cdi"

# 3) Install the NVIDIA DRA Driver for GPUs (Helm).
helm repo add nvidia https://nvidia.github.io/k8s-dra-driver-gpu
helm repo update
helm install nvidia-dra-driver-gpu nvidia/nvidia-dra-driver-gpu \
  --version 25.3.0 \
  --namespace nvidia-dra-driver-gpu --create-namespace \
  --set resources.gpus.enabled=true

# 4) Confirm the driver published its inventory as ResourceSlices.
kubectl get resourceslices
kubectl get deviceclasses


## Basic Usage

### Quick start

The everyday pattern is three objects:

1. A **`DeviceClass`** (usually installed by the driver — e.g. `gpu.nvidia.com`).
2. A **`ResourceClaimTemplate`** describing the device you want.
3. A **Pod** that references the template and exposes the claim to a container
   via `resources.claims`.

The manifest below requests a single NVIDIA GPU and runs `nvidia-smi`.

In [ ]:
# A minimal DRA request: one GPU, selected by DeviceClass, used by one pod.
manifest = '''
apiVersion: resource.k8s.io/v1
kind: ResourceClaimTemplate
metadata:
  name: single-gpu
  namespace: gpu-test
spec:
  spec:
    devices:
      requests:
        - name: gpu                 # request name, referenced by the container
          deviceClassName: gpu.nvidia.com
          allocationMode: ExactCount
          count: 1
---
apiVersion: v1
kind: Pod
metadata:
  name: gpu-smi
  namespace: gpu-test
spec:
  restartPolicy: Never
  resourceClaims:
    - name: gpu                     # pod-level claim, backed by the template
      resourceClaimTemplateName: single-gpu
  containers:
    - name: smi
      image: nvcr.io/nvidia/cuda:12.4.1-base-ubuntu22.04
      command: ["nvidia-smi", "-L"]
      resources:
        claims:
          - name: gpu               # bind the pod claim into this container
'''
print(manifest)

# Apply and inspect:
#   kubectl apply -f gpu-smi.yaml
#   kubectl get resourceclaims -n gpu-test          # see Allocated / ReservedFor
#   kubectl logs gpu-smi -n gpu-test                # -> the GPU's UUID


## Advanced Features

### 1. Attribute selectors (CEL)

A request can narrow devices with CEL expressions over the attributes a driver
publishes in its ResourceSlices. The NVIDIA driver exposes attributes such as
`productName`, memory `capacity`, and (for MIG) the profile, so you can demand
"an 80 GB GPU" or "an A100", not just "a GPU".

### 2. Sharing one claim across containers/pods

A single `ResourceClaim` (as opposed to a per-pod template) can be referenced by
several pods or containers, which makes them share the *same* underlying
device(s). Combined with the NVIDIA driver's `sharing` config (time-slicing or
MPS) this is how you give several light workloads one physical GPU.

### 3. Dynamic MIG and sharing strategies

The NVIDIA DRA driver accepts opaque vendor `config` on the claim to pick a
sharing strategy — `TimeSlicing`, `MPS`, or carving a GPU into **MIG** instances
on demand — instead of statically configuring each node.

### 4. Match attributes & constraints

`constraints` with `matchAttributes` force all selected devices to share an
attribute value — e.g. the same NVLink/IMEX domain — so a multi-GPU job lands on
GPUs that can actually talk over high-bandwidth links.

In [ ]:
# Two GPUs of the same model, in the same NVLink domain, time-sliced 4 ways.
claim = '''
apiVersion: resource.k8s.io/v1
kind: ResourceClaimTemplate
metadata:
  name: paired-gpus
  namespace: gpu-test
spec:
  spec:
    devices:
      requests:
        - name: gpus
          deviceClassName: gpu.nvidia.com
          allocationMode: ExactCount
          count: 2
          selectors:
            - cel:
                # Only A100/H100-class cards with >= 40 GB of memory.
                expression: >-
                  device.attributes["gpu.nvidia.com"].productName.contains("A100") &&
                  device.capacity["gpu.nvidia.com"].memory.compareTo(quantity("40Gi")) >= 0
      constraints:
        # Both selected GPUs must report the same NVLink clique / IMEX domain.
        - requests: ["gpus"]
          matchAttributes: ["gpu.nvidia.com/nvlinkDomain"]
      config:
        - requests: ["gpus"]
          opaque:
            driver: gpu.nvidia.com
            parameters:
              apiVersion: resource.nvidia.com/v1beta1
              kind: GpuConfig
              sharing:
                strategy: TimeSlicing          # or MPS
'''
print(claim)


## Use Cases

### Real-world Applications of DRA

#### Heterogeneous GPU clusters
- **Context:** A cluster mixes T4, A100, and H100 nodes; jobs must land on a card
  that fits the model.
- **Implementation:** Workloads request `gpu.nvidia.com` with a CEL selector on
  `productName`/`memory` instead of relying on node labels and `nodeSelector`.
- **Results:** Right-sized placement and higher utilization without per-node
  taints/labels micromanagement.

#### Fractional GPUs for inference / notebooks
- **Context:** Many small inference services or Jupyter pods waste whole GPUs.
- **Implementation:** A shared `ResourceClaim` plus NVIDIA `sharing: MPS` (or
  `TimeSlicing`) packs several pods onto one GPU.
- **Results:** Far better density for bursty, low-utilization workloads.

#### Dynamic MIG without static node config
- **Context:** A100/H100 capacity must flex between a few large jobs and many
  small ones during the day.
- **Implementation:** Claims request MIG profiles via the driver's opaque config;
  the driver creates/destroys MIG instances on demand.
- **Results:** No cluster-wide MIG geometry decision baked into nodes ahead of
  time.

#### Tightly-coupled multi-GPU / multi-node training
- **Context:** Large-model training needs GPUs that share a high-bandwidth NVLink
  domain (e.g. GB200 NVL72 via IMEX).
- **Implementation:** A multi-device claim with a `matchAttributes` constraint on
  the NVLink/IMEX domain.
- **Results:** The scheduler guarantees co-located, high-bandwidth GPUs for
  collective communication.

## Best Practices

1. **Prefer `ResourceClaimTemplate` for workloads.** It mints a fresh claim per
   pod, which is what you want for Deployments/Jobs; reserve a standalone
   `ResourceClaim` for the deliberate case of cross-pod sharing.
2. **Let admins own `DeviceClass`, users own claims.** Curate the device "menu"
   and default config centrally; keep workload manifests small and portable.
3. **Select by attribute, avoid node labels.** Express hardware needs as CEL over
   published attributes so manifests stay portable across node pools.
4. **Pin driver and chart versions.** DRA APIs and the NVIDIA driver evolve
   quickly; pin the Helm chart and verify the served `resource.k8s.io` version.
5. **Confirm CDI is enabled** on every GPU node — the NVIDIA DRA driver depends
   on it to inject devices.
6. **Keep claims scoped to what you'll use.** Over-broad `count`/selectors hurt
   schedulability; request exactly the devices the workload needs.
7. **Watch feature-gate status per cluster.** Core DRA is GA in 1.34, but
   admin-access, partitionable-devices, and consumable-capacity remain
   alpha/beta — don't depend on them on managed clusters that disable them.

## Common Pitfalls

1. **Forgetting the container binding.** Declaring `spec.resourceClaims` on the
   pod is not enough — each container must list the claim under
   `resources.claims`, or it won't see the device.
2. **Feature gate / API version mismatch.** On pre-1.34 clusters the gate must be
   on for *all* control-plane components and kubelet, and the right
   `resource.k8s.io` version served; otherwise claims sit `Pending`.
3. **CDI not enabled.** Without CDI mode in the NVIDIA Container Toolkit the
   driver's `NodePrepareResources` can't inject the GPU and pods fail to start.
4. **Confusing DRA with the device plugin.** Don't request `nvidia.com/gpu` *and*
   a DRA claim for the same GPU on the same node — pick one allocation path.
5. **Templates vs. shared claims.** Using one shared `ResourceClaim` where you
   meant per-pod isolation makes pods contend for the same device (or block).
6. **Over-constrained selectors.** CEL that no published device satisfies yields
   `UnschedulableAndUnresolvable`; check `ResourceSlice` attributes first.
7. **Alpha sprawl.** Building on admin-access or partitionable-device features
   that a target cluster doesn't enable leads to silently dropped behavior.

## Performance Optimization

### Optimizing DRA for production

#### Scheduling cost
- **Selector complexity:** very broad CEL selectors over large device pools make
  the scheduler evaluate more candidates per claim — keep expressions specific.
- **`ResourceSlice` size:** dense nodes (8× GPUs, many MIG slices) produce large
  slices; this is normal, but huge pools increase scheduler filtering work.

#### Sharing strategy choice
- **MPS** gives concurrent execution with spatial partitioning of SMs — best for
  several compute-bound co-tenants that tolerate shared memory bandwidth.
- **Time-slicing** is simpler and isolates nothing temporally beyond context
  switching — fine for bursty, latency-tolerant work; avoid for tight latency
  SLOs.
- **MIG** gives hardware-isolated partitions (memory + SMs) — pick it when
  tenants need predictable, isolated performance.

#### Allocation churn
- Dynamic MIG (de)partitioning has real cost; avoid thrashing profiles. Batch
  workloads of similar shape onto a stable geometry where possible.

In [ ]:
# Reason about how a sharing strategy maps a physical GPU onto N tenants.
def plan_sharing(strategy, tenants, gpu_mem_gb=80):
    strategy = strategy.lower()
    if strategy == "mig":
        # A100/H100 MIG profiles roughly halve memory per split step.
        profiles = {1: "1g.80gb", 2: "3g.40gb", 4: "2g.20gb", 7: "1g.10gb"}
        slices = min(profiles, key=lambda k: abs(k - tenants))
        return (f"MIG: ~{slices} hardware-isolated instances ({profiles[slices]}); "
                "isolated memory+SMs, predictable latency")
    if strategy == "mps":
        return (f"MPS: {tenants} concurrent contexts sharing {gpu_mem_gb} GB; "
                "spatial SM sharing, no hard memory isolation")
    if strategy == "timeslicing":
        return (f"TimeSlicing: {tenants} contexts round-robin the whole GPU; "
                "simplest, but tenants contend temporally")
    return "unknown strategy"

for s in ("MIG", "MPS", "TimeSlicing"):
    print(f"{s:12s} -> {plan_sharing(s, tenants=4)}")


## Production Deployment

### Deploying DRA in production

The DRA driver itself runs as a **DaemonSet** on GPU nodes (the Helm chart
handles this). Your job is to ship a curated `DeviceClass` and have workloads
consume it. Below: an admin-curated class restricting to a GPU family, and a
Deployment whose pods each get their own GPU claim via a template.

```yaml
# Admin-curated DeviceClass: only expose A100/H100-class GPUs under a friendly name.
apiVersion: resource.k8s.io/v1
kind: DeviceClass
metadata:
  name: datacenter-gpu
spec:
  selectors:
    - cel:
        expression: >-
          device.driver == "gpu.nvidia.com" &&
          device.attributes["gpu.nvidia.com"].productName.matches("A100|H100")
---
apiVersion: resource.k8s.io/v1
kind: ResourceClaimTemplate
metadata:
  name: one-datacenter-gpu
  namespace: ml-serving
spec:
  spec:
    devices:
      requests:
        - name: gpu
          deviceClassName: datacenter-gpu
          allocationMode: ExactCount
          count: 1
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: triton
  namespace: ml-serving
spec:
  replicas: 3
  selector: { matchLabels: { app: triton } }
  template:
    metadata: { labels: { app: triton } }
    spec:
      resourceClaims:
        - name: gpu
          resourceClaimTemplateName: one-datacenter-gpu   # one GPU claim per pod
      containers:
        - name: triton
          image: nvcr.io/nvidia/tritonserver:24.05-py3
          resources:
            claims:
              - name: gpu
```


## Monitoring and Observability

### Monitoring DRA in production

#### Key things to track

- **Claim lifecycle:** `kubectl get resourceclaims -A` — watch for claims stuck
  `Pending`/`WaitingForFirstConsumer` or pods `Unschedulable` because no device
  matches.
- **Device inventory:** `kubectl get resourceslices` — confirm every GPU node's
  driver is publishing the expected devices and attributes.
- **Allocation pressure:** how many devices in each pool are `ReservedFor`
  (allocated) vs. free — your "are we out of GPUs?" signal.
- **GPU telemetry:** pair DRA with **DCGM Exporter** for utilization, memory,
  power, temperature, and Tensor-Core activity of the allocated devices.

#### Useful commands & logs

- `kubectl describe resourceclaim <name>` — shows `status.allocation` and the
  scheduling result / reason if unallocated.
- `kubectl describe pod <pod>` — scheduler events explain
  `UnschedulableAndUnresolvable` device requests.
- DRA driver DaemonSet logs (`-n nvidia-dra-driver-gpu`) — `NodePrepareResources`
  errors (CDI, MIG creation) surface here.
- Scheduler logs — DRA filter/allocation decisions when claims won't bind.

## Troubleshooting

### Common Issues with DRA

#### Pod stuck `Pending`, claim never allocated
**Symptoms:** `kubectl get resourceclaims` shows no allocation; pod events read
`UnschedulableAndUnresolvable`.

**Cause:** No `ResourceSlice` advertises a device matching the request's CEL
selectors (wrong attribute name, too-strict memory bound), or the driver isn't
running on any eligible node.

**Solution:** Inspect `kubectl get resourceslices -o yaml` to see real attribute
keys/values, loosen the selector, and confirm the DRA driver DaemonSet is healthy
on GPU nodes.

#### Claims `Pending` on a pre-1.34 cluster
**Symptoms:** `resource.k8s.io` resources exist but nothing schedules.

**Cause:** The `DynamicResourceAllocation` feature gate is off on one of the
control-plane components or the kubelet, or the API version isn't served.

**Solution:** Enable the gate on the API server, scheduler, controller-manager,
*and* kubelet, set `--runtime-config=resource.k8s.io/v1=true`, and restart.

#### Pod schedules but container can't see the GPU
**Symptoms:** `nvidia-smi` in the container fails or shows no devices.

**Cause:** CDI isn't enabled in the NVIDIA Container Toolkit, or the container
omits `resources.claims`, so the device was never injected.

**Solution:** Set the toolkit runtime `mode = "cdi"`, and ensure every container
that needs the device lists the claim under `resources.claims`.

#### MIG / sharing config ignored
**Symptoms:** Requested MPS/MIG behavior doesn't take effect.

**Cause:** The opaque `config` `driver`/`apiVersion`/`kind` don't match what the
NVIDIA DRA driver expects, or the GPU doesn't support the requested mode.

**Solution:** Match the driver's documented `GpuConfig`/`MigDeviceConfig` schema
and verify the hardware supports MIG/MPS.

## Comparison with Alternatives

### How DRA Compares to Other Solutions

| Aspect | DRA + NVIDIA DRA driver | Device plugin (`nvidia.com/gpu`) | Static time-slicing (device plugin) | Static MIG (GPU Operator) |
|---|---|---|---|---|
| Selection model | Attributes via CEL | Integer count only | Integer count only | Integer count of fixed slices |
| GPU sharing | Per-claim TimeSlicing / MPS / dynamic MIG | None (whole GPU) | Cluster/node-wide replicas | Pre-partitioned, node-wide |
| Topology awareness | `matchAttributes` (e.g. NVLink domain) | None | None | None |
| Config location | Travels with the claim | Node/cluster config | Node/cluster config | Node/cluster config |
| Heterogeneous clusters | First-class | Needs node labels | Needs node labels | Needs node labels |
| Maturity | GA in K8s 1.34 (driver evolving) | Mature, ubiquitous | Mature | Mature |

### When to Choose DRA

Choose DRA when:

- You need **attribute-based** device selection across **heterogeneous** GPUs.
- You want **per-workload** sharing (MPS / time-slicing / dynamic MIG) rather
  than baking a single mode into every node.
- You run **topology-sensitive** multi-GPU jobs needing NVLink/IMEX co-location.

Stick with the classic **device plugin** when your nodes are homogeneous, you
allocate whole GPUs, and you value the most mature, widely-supported path.

## Resources

### Official Documentation

- Kubernetes — Dynamic Resource Allocation:
  https://kubernetes.io/docs/concepts/scheduling-eviction/dynamic-resource-allocation/
- Kubernetes — DRA API (`resource.k8s.io`) reference:
  https://kubernetes.io/docs/reference/kubernetes-api/cluster-resources/
- NVIDIA DRA Driver for GPUs (GitHub):
  https://github.com/NVIDIA/k8s-dra-driver-gpu
- NVIDIA GPU Operator docs:
  https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/

### Tutorials and Guides

- KEP-4381 — DRA with structured parameters:
  https://github.com/kubernetes/enhancements/tree/master/keps/sig-node/4381-dra-structured-parameters
- NVIDIA blog — Dynamic GPU allocation on Kubernetes with DRA:
  https://developer.nvidia.com/blog/
- Kubernetes blog — DRA feature updates by release:
  https://kubernetes.io/blog/

### Community Resources

- Kubernetes SIG-Node (DRA owner):
  https://github.com/kubernetes/community/tree/master/sig-node
- Kubernetes Slack `#sig-node` / `#wg-device-management`:
  https://kubernetes.slack.com
- NVIDIA k8s-dra-driver-gpu issues:
  https://github.com/NVIDIA/k8s-dra-driver-gpu/issues

### Related Technologies

- Container Device Interface (CDI) — how devices are injected into containers
- NVIDIA Device Plugin for Kubernetes — the classic allocation path
- Multi-Instance GPU (MIG), MPS, and time-slicing — sharing mechanisms
- DCGM Exporter — telemetry for the GPUs DRA hands out